# Deep Q-Networks

Q-Learning with a neural network in place of the table, stabilized by experience replay and a target network.

This notebook comes from the [Deep Q-Networks](https://pabl0aranda.github.io/Atlas-ML/algorithms/deep-q-networks/) page on Atlas ML, an interactive guide to machine learning. Run the cells from top to bottom.

## Code from the page

In [ ]:
import torch
from torch import nn

def train_step(online, target, optimizer, batch, gamma=0.98):
    states, actions, rewards, next_states, done = batch
    predicted = online(states).gather(1, actions.unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        future = target(next_states).max(dim=1).values
        goal = rewards + gamma * future * (1 - done)
    loss = nn.functional.smooth_l1_loss(predicted, goal)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

## Try it

In [ ]:
# One training step on a random batch of 8 transitions, with 4 state values and 2 actions.
online, target = nn.Linear(4, 2), nn.Linear(4, 2)
target.load_state_dict(online.state_dict())
optimizer = torch.optim.Adam(online.parameters(), lr=1e-3)
batch = (torch.randn(8, 4), torch.randint(0, 2, (8,)), torch.randn(8), torch.randn(8, 4), torch.zeros(8))
print(train_step(online, target, optimizer, batch))